# Analysis — CPU vs GPU Speedup

Loads `results/timings_cpu.csv` and `results/timings_gpu.csv`, computes medians,
and produces all tables and plots used in the report. Run this after both sweep notebooks complete.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

# Phase display order
PHASE_ORDER = ['fe_train', 'fe_test', 'lr_fit', 'lr_predict', 'xgb_fit', 'xgb_predict']
PHASE_LABEL = {
    'fe_train':    'FE Train',
    'fe_test':     'FE Test',
    'lr_fit':      'LR Fit',
    'lr_predict':  'LR Predict',
    'xgb_fit':     'XGB Fit',
    'xgb_predict': 'XGB Predict'
}
FRACS = [0.10, 0.25, 0.50, 0.75, 1.00]


In [ ]:
cpu = pd.read_csv('results/timings_cpu.csv')
gpu = pd.read_csv('results/timings_gpu.csv')

print(f'CPU rows: {len(cpu)}  GPU rows: {len(gpu)}')
assert len(cpu) == 90 and len(gpu) == 90, 'Expected 90 rows each (3 runs × 5 fracs × 6 phases)'

# Median per (device, fraction, phase)
cpu_med = cpu.groupby(['fraction', 'phase'])['seconds'].median().rename('cpu_s')
gpu_med = gpu.groupby(['fraction', 'phase'])['seconds'].median().rename('gpu_s')

df = pd.concat([cpu_med, gpu_med], axis=1).reset_index()
df['speedup'] = df['cpu_s'] / df['gpu_s']
df['phase']   = pd.Categorical(df['phase'], categories=PHASE_ORDER, ordered=True)
df = df.sort_values(['fraction', 'phase'])
print(df)


In [ ]:
# ── Main speedup table (5 fractions × 6 phases) ──────────────────────────────
pivot = df.pivot(index='fraction', columns='phase', values='speedup')[PHASE_ORDER]
pivot.index = [f'{int(f*100)}%' for f in pivot.index]
pivot.columns = [PHASE_LABEL[c] for c in pivot.columns]
print("=== Speedup Table (CPU time / GPU time, median of 3 runs) ===")
print(pivot.round(2).to_string())


In [ ]:
# ── Speedup-vs-fraction plot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

colors = plt.cm.tab10.colors
pct = [f'{int(f*100)}%' for f in FRACS]

for i, phase in enumerate(PHASE_ORDER):
    vals = [df.loc[(df['fraction'] == f) & (df['phase'] == phase), 'speedup'].iloc[0]
            for f in FRACS]
    ax.plot(pct, vals, marker='o', label=PHASE_LABEL[phase], color=colors[i], linewidth=2)

ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, label='Break-even (1×)')
ax.set_xlabel('Training set fraction')
ax.set_ylabel('Speedup (CPU time / GPU time)')
ax.set_title('GPU Speedup vs Dataset Size — CFPB Complaints Pipeline')
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/speedup_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/speedup_by_phase.png')


In [ ]:
# ── Stacked bar: total pipeline time CPU vs GPU ───────────────────────────────
cpu_totals = cpu.groupby(['fraction', 'phase'])['seconds'].median().unstack()[PHASE_ORDER]
gpu_totals = gpu.groupby(['fraction', 'phase'])['seconds'].median().unstack()[PHASE_ORDER]

pct = [f'{int(f*100)}%' for f in FRACS]
x   = np.arange(len(FRACS))
w   = 0.35
colors6 = plt.cm.Set2.colors[:6]

fig, ax = plt.subplots(figsize=(10, 5))
bottom_cpu = np.zeros(len(FRACS))
bottom_gpu = np.zeros(len(FRACS))

for i, phase in enumerate(PHASE_ORDER):
    cpu_vals = cpu_totals[phase].values
    gpu_vals = gpu_totals[phase].values
    ax.bar(x - w/2, cpu_vals, w, bottom=bottom_cpu, color=colors6[i],
           label=PHASE_LABEL[phase] if i == 0 else '', alpha=0.85)
    ax.bar(x + w/2, gpu_vals, w, bottom=bottom_gpu, color=colors6[i], alpha=0.5)
    bottom_cpu += cpu_vals
    bottom_gpu += gpu_vals

# Legend and labels
from matplotlib.patches import Patch
legend_phases = [Patch(facecolor=colors6[i], label=PHASE_LABEL[p]) for i, p in enumerate(PHASE_ORDER)]
legend_device = [Patch(facecolor='grey', alpha=0.85, label='CPU'),
                 Patch(facecolor='grey', alpha=0.5,  label='GPU')]
ax.legend(handles=legend_phases + legend_device, fontsize=8, loc='upper left', ncol=2)
ax.set_xticks(x); ax.set_xticklabels(pct)
ax.set_xlabel('Training set fraction')
ax.set_ylabel('Total wall-clock time (s)')
ax.set_title('Total Pipeline Time — CPU vs GPU')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/pipeline_time_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/pipeline_time_bar.png')


In [ ]:
# ── CV context numbers (from best_params.json) ────────────────────────────────
with open('results/best_params.json') as f:
    bp = json.load(f)

lr_cpu_cv  = bp['cv_context_seconds']['lr_gridsearch']
xgb_cpu_cv = bp['cv_context_seconds']['xgb_randomizedsearch']

# GPU XGB CV time is stored in the GPU notebook as t_xgb_cv_gpu
# If you ran both notebooks, paste the GPU CV time here:
xgb_gpu_cv = None  # e.g. 240.0  — fill in from GPU_Solution output

print("=== Full CV Context (100% data, 1 run) ===")
print(f"LR  GridSearchCV  (50 fits)  — CPU: {lr_cpu_cv:.0f}s  ({lr_cpu_cv/60:.1f} min)  |  GPU: N/A (sklearn LR)")
if xgb_gpu_cv:
    print(f"XGB RandomizedCV (300 fits) — CPU: {xgb_cpu_cv:.0f}s ({xgb_cpu_cv/60:.1f} min) | GPU: {xgb_gpu_cv:.0f}s ({xgb_gpu_cv/60:.1f} min) | Speedup: {xgb_cpu_cv/xgb_gpu_cv:.1f}x")
else:
    print(f"XGB RandomizedCV (300 fits) — CPU: {xgb_cpu_cv:.0f}s ({xgb_cpu_cv/60:.1f} min) | GPU: [fill in xgb_gpu_cv above]")


In [ ]:
# ── Print full-size summary for copy-paste into report ───────────────────────
full = df[df['fraction'] == 1.00].set_index('phase').loc[PHASE_ORDER]
print("=== 100% Dataset Summary ===")
print(f"{'Phase':<14} {'CPU (s)':>10} {'GPU (s)':>10} {'Speedup':>10}")
print('-' * 46)
for phase, row in full.iterrows():
    print(f"{PHASE_LABEL[phase]:<14} {row['cpu_s']:>10.3f} {row['gpu_s']:>10.3f} {row['speedup']:>10.2f}x")
